# Module 03: API for Accessing Large Language Models

**CareerAlign GenAI Course**

---

## Overview

Every production GenAI system communicates with language models through HTTP APIs. This notebook takes you from raw API requests through provider SDKs, streaming, cost management, and structured outputs.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand the Chat Completions interface** -- the universal contract between your code and an LLM
2. **Use the OpenAI Python SDK** to make synchronous and asynchronous API calls
3. **Construct messages arrays** with system, user, and assistant roles
4. **Stream responses** token-by-token for responsive user experiences
5. **Implement function calling / tool use** to let models invoke structured actions
6. **Extract structured outputs** using `response_format` (JSON mode) and Pydantic
7. **Track token usage and estimate costs** before and after API calls
8. **Handle errors with automatic retries** using exponential backoff
9. **Make async API calls** for high-throughput applications
10. **Manage multi-turn conversations** and compare generation parameters

## Prerequisites

- An OpenAI API key (set as `OPENAI_API_KEY` environment variable)
- Python packages: `openai`, `tiktoken`, `tenacity`, `pydantic`

---

## 0. Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install openai tiktoken tenacity pydantic

In [ ]:
import os
import json
import time
from pprint import pprint

# Ensure the API key is set
assert os.environ["OPENAI_API_KEY"], "Please set the OPENAI_API_KEY environment variable."

from openai import OpenAI

# Initialize the client -- reads OPENAI_API_KEY from the environment automatically
client = OpenAI()

print("OpenAI client initialized successfully.")

---

## 1. Basic Chat Completion Call

The Chat Completions interface is the universal contract between your application code and a large language model. You construct an array of **messages** representing a conversation, send it along with configuration parameters to an HTTP endpoint, and receive back the model's generated text plus metadata.

Key points:
- LLM APIs are **stateless** -- each request is independent; the provider maintains zero conversation state between requests.
- Your code is entirely responsible for maintaining conversation history.
- The `model` field specifies which model to use (e.g., `gpt-4o`, `gpt-4o-mini`).

In [ ]:
# A minimal chat completion request
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is an API? Explain in two sentences."}
    ],
    temperature=0.0,
    max_tokens=256,
)

# Extract the generated text
print("Response:")
print(response.choices[0].message.content)
print()

# Inspect the finish reason
print(f"Finish reason: {response.choices[0].finish_reason}")
# 'stop' = natural completion, 'length' = hit max_tokens (truncated)

---

## 2. System / User / Assistant Message Roles

Each message in the array has a **role**:

| Role | Purpose |
|---|---|
| `system` | Sets persona, behavioral constraints, and task instructions. Carries persistent instructional weight. |
| `user` | Represents the human (or the application acting on behalf of the human). |
| `assistant` | Previous model responses -- included for multi-turn context so the model stays consistent. |

A detailed, specific system message is the single highest-leverage knob for controlling output quality.

In [ ]:
# Demonstrating all three roles
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        # SYSTEM: sets the model's persona and constraints
        {
            "role": "system",
            "content": (
                "You are a senior Python developer. "
                "Explain concepts with concise code examples. "
                "Always include type hints in your code."
            ),
        },
        # USER: the question
        {"role": "user", "content": "What is a decorator in Python?"},
    ],
    temperature=0.0,
    max_tokens=512,
)

print(response.choices[0].message.content)

In [ ]:
# Including an assistant message for context (few-shot / multi-turn)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful translator. Translate English to French."},
        # Few-shot example: user asks, assistant demonstrates the expected format
        {"role": "user", "content": "Hello, how are you?"},
        {"role": "assistant", "content": "Bonjour, comment allez-vous?"},
        # The real request
        {"role": "user", "content": "The weather is beautiful today."},
    ],
    temperature=0.0,
    max_tokens=128,
)

print("Translation:", response.choices[0].message.content)

---

## 3. Streaming Responses

Without streaming, the user waits 5-30 seconds for the full response. Streaming delivers tokens as they are generated, so text appears almost immediately (time-to-first-token is typically 200-500ms).

The protocol is **Server-Sent Events (SSE)**: you set `stream=True`, and the SDK returns an iterator of chunks. Each chunk contains a `delta` with the incremental content.

In [ ]:
# Streaming a response token-by-token
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise technical writer."},
        {"role": "user", "content": "Explain how HTTPS works in 3 bullet points."},
    ],
    temperature=0.0,
    max_tokens=512,
    stream=True,  # Enable streaming
)

# Accumulate the full response while printing tokens as they arrive
full_text = ""
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content:
        full_text += delta.content
        print(delta.content, end="", flush=True)

print("\n\n--- Streaming complete ---")
print(f"Total length: {len(full_text)} characters")

---

## 4. Function Calling / Tool Use

Function calling (also called "tool use") allows the model to request that your code execute a specific function with structured arguments. The model does **not** execute the function itself -- it generates a structured JSON request, and your code decides whether and how to execute it.

The flow is:
1. You define available tools (functions) with names, descriptions, and parameter schemas.
2. The model may respond with a `tool_calls` request instead of (or alongside) text.
3. Your code executes the function and sends the result back as a `tool` role message.
4. The model uses the result to formulate its final response.

In [ ]:
import json

# Step 1: Define the tools (functions) available to the model
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g. 'San Francisco'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["city"],
            },
        },
    }
]


# Step 2: A mock function that simulates real execution
def get_weather(city: str, unit: str = "celsius") -> dict:
    """Simulated weather lookup."""
    weather_data = {
        "San Francisco": {"temp": 18, "condition": "Foggy"},
        "New York": {"temp": 25, "condition": "Sunny"},
        "London": {"temp": 14, "condition": "Rainy"},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown"})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9 / 5 + 32)
    data["unit"] = unit
    data["city"] = city
    return data


# Step 3: Send the initial request with tools
messages = [
    {"role": "system", "content": "You are a helpful assistant with access to weather data."},
    {"role": "user", "content": "What's the weather like in San Francisco and New York?"},
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",  # Let the model decide when to call tools
    temperature=0.0,
    max_tokens=1024,
)

assistant_message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Tool calls requested: {len(assistant_message.tool_calls) if assistant_message.tool_calls else 0}")

In [ ]:
# Step 4: Execute the tool calls and send results back to the model
if assistant_message.tool_calls:
    # Add the assistant's tool-call message to the conversation
    messages.append(assistant_message)

    # Execute each tool call
    for tool_call in assistant_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

        print(f"Calling {function_name}({function_args})")

        # Dispatch to the actual function
        if function_name == "get_weather":
            result = get_weather(**function_args)
        else:
            result = {"error": f"Unknown function: {function_name}"}

        # Add the tool result to the conversation
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            }
        )

    # Step 5: Get the model's final response using the tool results
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.0,
        max_tokens=1024,
    )

    print("\nFinal response:")
    print(final_response.choices[0].message.content)

---

## 5. Structured Output with `response_format` (JSON Mode)

OpenAI provides a native `response_format` parameter that constrains generation at the token level. When you provide a JSON schema, the model **physically cannot** produce output that violates the schema. This is not post-hoc validation -- it is built into the generation process.

Two modes:
- `{"type": "json_object"}` -- guarantees valid JSON output (you describe the shape in the prompt)
- `{"type": "json_schema", "json_schema": {...}}` -- guarantees output conforming to an exact schema

In [ ]:
# JSON mode: guarantees valid JSON, but the shape is controlled by the prompt
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a structured data extractor. "
                "Return a JSON object with keys: 'frameworks' (list of objects with 'name' and 'description')."
            ),
        },
        {"role": "user", "content": "List 3 popular Python web frameworks."},
    ],
    response_format={"type": "json_object"},
    temperature=0.0,
    max_tokens=512,
)

# Parse the guaranteed-valid JSON
data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2))

In [ ]:
# Strict JSON Schema mode: enforces exact schema compliance
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract entities from the text provided by the user."},
        {
            "role": "user",
            "content": (
                "Anthropic, based in San Francisco, announced on March 4, 2025 "
                "that CEO Dario Amodei will present new safety research at "
                "the United Nations headquarters in New York."
            ),
        },
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "entity_extraction",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "entities": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string"},
                                "entity_type": {
                                    "type": "string",
                                    "enum": ["person", "organization", "location", "date"],
                                },
                            },
                            "required": ["name", "entity_type"],
                            "additionalProperties": False,
                        },
                    },
                    "summary": {"type": "string"},
                },
                "required": ["entities", "summary"],
                "additionalProperties": False,
            },
        },
    },
    temperature=0.0,
    max_tokens=512,
)

result = json.loads(response.choices[0].message.content)
print("Extracted entities:")
for entity in result["entities"]:
    print(f"  - {entity['name']} ({entity['entity_type']})")
print(f"\nSummary: {result['summary']}")

### Structured Outputs with Pydantic

For a more Pythonic approach, you can define Pydantic models and let the SDK convert them to JSON schemas automatically. This gives you type-safe parsing and IDE autocompletion on the result.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal


# Define the output structure as a Pydantic model
class Entity(BaseModel):
    name: str = Field(description="The entity name")
    entity_type: Literal["person", "organization", "location", "date"] = Field(
        description="The category of the entity"
    )


class ExtractionResult(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning before extraction")
    entities: List[Entity] = Field(description="List of extracted entities")
    summary: str = Field(description="One-sentence summary of the text")


# Use the beta parse method with Pydantic
completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract entities from the given text."},
        {
            "role": "user",
            "content": (
                "On January 15, 2025, OpenAI released a new model from their "
                "San Francisco headquarters. CEO Sam Altman announced the launch at CES in Las Vegas."
            ),
        },
    ],
    response_format=ExtractionResult,
    temperature=0.0,
)

# The result is a validated Pydantic object
result = completion.choices[0].message.parsed
print(f"Reasoning: {result.reasoning}\n")
for e in result.entities:
    print(f"  {e.name} -> {e.entity_type}")
print(f"\nSummary: {result.summary}")

---

## 6. Token Usage Tracking and Cost Estimation

LLM APIs charge per token, with separate rates for input and output tokens. Output tokens are typically 3-5x more expensive than input tokens.

Key pricing (per 1M tokens, early 2025):

| Model | Input | Output | Context |
|---|---|---|---|
| `gpt-4o` | $2.50 | $10.00 | 128K |
| `gpt-4o-mini` | $0.15 | $0.60 | 128K |
| `claude-sonnet-4-20250514` | $3.00 | $15.00 | 200K |
| `gemini-2.0-flash` | $0.10 | $0.40 | 1M |

A token is roughly 3/4 of an English word, or about 4 characters.

In [ ]:
# Make a call and inspect the usage metadata
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain the difference between REST and GraphQL APIs."},
    ],
    temperature=0.0,
    max_tokens=512,
)

# Token usage from the response
usage = response.usage
print(f"Prompt (input) tokens:     {usage.prompt_tokens}")
print(f"Completion (output) tokens: {usage.completion_tokens}")
print(f"Total tokens:              {usage.total_tokens}")

# Calculate cost for gpt-4o-mini
INPUT_RATE = 0.15   # per 1M tokens
OUTPUT_RATE = 0.60  # per 1M tokens

cost_input = (usage.prompt_tokens / 1_000_000) * INPUT_RATE
cost_output = (usage.completion_tokens / 1_000_000) * OUTPUT_RATE
total_cost = cost_input + cost_output

print(f"\nEstimated cost (gpt-4o-mini): ${total_cost:.6f}")

In [ ]:
import tiktoken


# Pre-request token counting with tiktoken
PRICING = {
    "gpt-4o":      (2.50, 10.00),   # (input, output) per 1M tokens
    "gpt-4o-mini": (0.15,  0.60),
}


def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """Count tokens for an OpenAI model using tiktoken."""
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))


def estimate_cost(
    messages: list,
    model: str = "gpt-4o",
    est_output_tokens: int = 500,
) -> dict:
    """Estimate cost before sending a request."""
    enc = tiktoken.encoding_for_model(model)
    input_tokens = sum(
        len(enc.encode(m["content"])) + 4  # +4 for role/message overhead
        for m in messages
    )
    rate_in, rate_out = PRICING.get(model, (2.50, 10.00))
    cost_in = (input_tokens / 1_000_000) * rate_in
    cost_out = (est_output_tokens / 1_000_000) * rate_out
    return {
        "input_tokens": input_tokens,
        "est_output_tokens": est_output_tokens,
        "est_cost_usd": round(cost_in + cost_out, 6),
    }


# Example: estimate before calling
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a 500-word essay about climate change."},
]

estimate = estimate_cost(messages, model="gpt-4o", est_output_tokens=700)
print("Pre-request cost estimate:")
pprint(estimate)

---

## 7. Error Handling and Retries

LLM APIs are inherently unreliable at scale. Common transient failures include:
- **Rate limiting (429)** -- you exceeded requests/minute or tokens/minute
- **Server errors (500, 502, 503)** -- provider outage or high load
- **Network errors** -- DNS, timeout, or TLS failures

**Never retry** 400 Bad Request or 401 Unauthorized -- these are bugs in your code, not transient failures.

The `tenacity` library provides robust retry logic with exponential backoff and jitter.

In [ ]:
import openai
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)


@retry(
    retry=retry_if_exception_type(
        (
            openai.RateLimitError,       # 429
            openai.APIConnectionError,   # network errors
            openai.InternalServerError,  # 500
        )
    ),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    stop=stop_after_attempt(5),
)
def robust_completion(messages: list, model: str = "gpt-4o-mini") -> str:
    """Call LLM with automatic retry on transient failures."""
    client = OpenAI()
    try:
        rsp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0,
            max_tokens=1024,
        )
        if rsp.choices[0].finish_reason == "length":
            raise ValueError("Response truncated -- increase max_tokens")
        return rsp.choices[0].message.content
    except openai.BadRequestError as e:
        # Don't retry 400s -- the request itself is wrong
        raise ValueError(f"Bad request: {e}") from e


# Test the robust function
result = robust_completion(
    messages=[{"role": "user", "content": "Say hello in 5 languages."}]
)
print(result)

---

## 8. Async API Calls

For web servers and high-throughput pipelines, async clients are essential. The `AsyncOpenAI` client uses async httpx under the hood, yielding control to the event loop while waiting for network I/O. This allows a single Python process to handle hundreds of concurrent LLM requests without threading.

**Always use the async client in async code** (e.g., FastAPI endpoints). Using the sync client in an async context blocks the entire event loop.

In [ ]:
import asyncio
from openai import AsyncOpenAI


async def async_completion(prompt: str) -> str:
    """Make an async chat completion call."""
    aclient = AsyncOpenAI()  # reads OPENAI_API_KEY from env
    response = await aclient.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=256,
    )
    return response.choices[0].message.content


async def run_concurrent_calls():
    """Run multiple LLM calls concurrently with asyncio.gather."""
    prompts = [
        "What is Python? (one sentence)",
        "What is JavaScript? (one sentence)",
        "What is Rust? (one sentence)",
    ]

    start = time.time()
    # All three calls run concurrently
    results = await asyncio.gather(*[async_completion(p) for p in prompts])
    elapsed = time.time() - start

    for prompt, result in zip(prompts, results):
        print(f"Q: {prompt}")
        print(f"A: {result}\n")

    print(f"All 3 calls completed in {elapsed:.2f}s (concurrent, not sequential)")


# In Jupyter, the event loop is already running, so use await directly
await run_concurrent_calls()

In [ ]:
# Async streaming example
async def async_stream(prompt: str) -> str:
    """Stream a response asynchronously."""
    aclient = AsyncOpenAI()
    stream = await aclient.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        max_tokens=512,
    )

    full_text = ""
    async for chunk in stream:
        delta = chunk.choices[0].delta
        if delta.content:
            full_text += delta.content
            print(delta.content, end="", flush=True)
    print()
    return full_text


result = await async_stream("Name 5 design patterns in software engineering, briefly.")

---

## 9. Multi-Turn Conversation Management

Since LLM APIs are stateless, **your code** must maintain the conversation history. Each request must include the full message history that the model needs to understand context.

For long conversations, you need strategies to manage context window limits:
- **Sliding window**: keep only the most recent N messages
- **Summary compression**: periodically summarize older messages
- **Semantic truncation**: remove messages least relevant to the current query

In [ ]:
class Conversation:
    """A simple multi-turn conversation manager."""

    def __init__(
        self,
        system_prompt: str,
        model: str = "gpt-4o-mini",
        max_history: int = 20,
    ):
        self.model = model
        self.max_history = max_history
        self.system_message = {"role": "system", "content": system_prompt}
        self.history: list[dict] = []
        self.total_tokens_used = 0

    def chat(self, user_message: str) -> str:
        """Send a message and get a response, maintaining conversation history."""
        # Add the user message
        self.history.append({"role": "user", "content": user_message})

        # Sliding window: trim old messages if needed
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history :]

        # Build the full messages array
        messages = [self.system_message] + self.history

        # Call the API
        response = client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.3,
            max_tokens=512,
        )

        assistant_content = response.choices[0].message.content
        self.total_tokens_used += response.usage.total_tokens

        # Add the assistant response to history
        self.history.append({"role": "assistant", "content": assistant_content})

        return assistant_content

    def get_stats(self) -> dict:
        return {
            "turns": len(self.history) // 2,
            "total_messages": len(self.history),
            "total_tokens_used": self.total_tokens_used,
        }


# Demo: multi-turn conversation
convo = Conversation(
    system_prompt="You are a knowledgeable Python tutor. Keep answers concise (2-3 sentences)."
)

# Turn 1
print("User: What is a list comprehension?")
print(f"AI: {convo.chat('What is a list comprehension?')}\n")

# Turn 2 (model remembers context from turn 1)
print("User: Can you show me an example with filtering?")
print(f"AI: {convo.chat('Can you show me an example with filtering?')}\n")

# Turn 3 (model has full context)
print("User: How does that compare to a generator expression?")
print(f"AI: {convo.chat('How does that compare to a generator expression?')}\n")

print("Conversation stats:")
pprint(convo.get_stats())

---

## 10. Comparing API Parameters: temperature, max_tokens, top_p

Key generation parameters:

| Parameter | Range | Effect |
|---|---|---|
| `temperature` | 0.0 - 2.0 | Controls randomness. 0 = deterministic, 1+ = creative. |
| `max_tokens` | 1 - context limit | Hard cap on output length. Always set generously to avoid truncation. |
| `top_p` | 0.0 - 1.0 | Nucleus sampling: limits to tokens in top p% probability mass. |
| `frequency_penalty` | -2.0 - 2.0 | Penalizes token repetition (OpenAI-specific). |
| `presence_penalty` | -2.0 - 2.0 | Encourages new topics (OpenAI-specific). |

**Best practice**: Use either `temperature` or `top_p`, not both. Their effects overlap, and combining them can produce unpredictable behavior.

In [ ]:
# Compare different temperature values
prompt = "Write a one-sentence tagline for a coffee shop."

print("=" * 60)
print("TEMPERATURE COMPARISON")
print("=" * 60)

for temp in [0.0, 0.5, 1.0, 1.5]:
    responses = []
    for _ in range(3):  # Generate 3 samples at each temperature
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=64,
        )
        responses.append(response.choices[0].message.content.strip())

    print(f"\nTemperature = {temp}:")
    for i, r in enumerate(responses, 1):
        print(f"  {i}. {r}")

In [ ]:
# Demonstrate max_tokens truncation
print("=" * 60)
print("MAX_TOKENS COMPARISON")
print("=" * 60)

for max_tok in [20, 50, 200]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Explain how a CPU works."}],
        temperature=0.0,
        max_tokens=max_tok,
    )
    text = response.choices[0].message.content
    reason = response.choices[0].finish_reason
    tokens_used = response.usage.completion_tokens

    print(f"\nmax_tokens={max_tok} | finish_reason={reason} | tokens_used={tokens_used}")
    print(f"  {text[:120]}{'...' if len(text) > 120 else ''}")

In [ ]:
# Compare top_p values
print("=" * 60)
print("TOP_P COMPARISON")
print("=" * 60)

for top_p_val in [0.1, 0.5, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Suggest a creative name for a tech startup."}],
        temperature=1.0,  # Keep temperature fixed
        top_p=top_p_val,
        max_tokens=64,
    )
    text = response.choices[0].message.content.strip()
    print(f"\ntop_p={top_p_val}: {text}")

---

## Summary

In this notebook we covered the core mechanics of working with LLM APIs:

| Topic | Key Takeaway |
|---|---|
| **Chat Completions** | Universal interface: messages array + config in, text + metadata out |
| **Message Roles** | `system` sets behavior, `user` provides input, `assistant` gives context |
| **Streaming** | Delivers tokens incrementally; critical for responsive UX |
| **Function Calling** | Model generates structured JSON to request function execution |
| **Structured Outputs** | `response_format` with JSON schema guarantees valid, parseable output |
| **Token Tracking** | Always monitor `usage` in responses; use `tiktoken` for pre-call estimates |
| **Error Handling** | Retry 429/500/network errors with exponential backoff; never retry 400/401 |
| **Async Calls** | Use `AsyncOpenAI` in async code to handle concurrent requests efficiently |
| **Multi-Turn** | Your code manages conversation state; use sliding window for long chats |
| **Parameters** | `temperature=0` for factual tasks, 0.7-1.0 for creative; set `max_tokens` generously |

### Next Steps

- **Module 04**: Fine-Tuning -- customize models for your specific domain
- **Module 06**: Prompt Engineering -- master the art of crafting effective prompts
- **Module 07**: RAG Systems -- combine APIs with retrieval for knowledge-grounded generation